# 👗 Garment Image Beautifier (nano banana)

Beautifies garment photos in bulk with Google's **Gemini 2.5 Flash Image** ("nano banana").

Reads images from the `input/` folder of your **GitHub** repo and saves the beautified versions back into the `output/` folder — no Google Drive, no pop-ups.

**How to use it:** run each cell in order (click the ▶️ on the left, top to bottom). You'll paste two things: your Gemini API key, and a GitHub token so it can save results.

## Step 1 — Install the tools
Run once. ~20 seconds.

In [ ]:
!pip install -q google-genai pillow requests
print('✅ Done installing.')

## Step 2 — Paste your Gemini API key
Get one at https://aistudio.google.com/apikey. A box appears when you run this — paste your key there (it stays hidden).

In [ ]:
from getpass import getpass
API_KEY = getpass('Paste your Gemini API key and press Enter: ')
print('✅ Gemini key saved for this session.')

## Step 3 — Paste your GitHub token
This lets the notebook save beautified images back into your repo's `output/` folder.

**To make one:** GitHub → Settings → Developer settings → **Fine-grained personal access tokens** → Generate new token →
- Repository access: **Only select repositories** → pick `garment-images`
- Permissions → Repository permissions → **Contents: Read and write**
- Generate, then copy the token (starts with `github_pat_`).

Paste it in the box when you run this cell.

In [ ]:
GITHUB_TOKEN = getpass('Paste your GitHub token and press Enter: ')
print('✅ GitHub token saved for this session.')

## Step 4 — Your prompt & settings
Your prompt is already loaded. Change the repo only if you use a different one.

In [ ]:
# 👇 YOUR PROMPT (loaded and ready — you can edit it here anytime)
PROMPT = """
A professional e-commerce product photograph of the garment shown in the provided image, presented in a 1:1 square aspect ratio. The item is positioned perfectly dead-center with a generous, uniform margin of approximately 10% to 15% negative space on the top, bottom, left, and right sides, ensuring the entire garment is safely contained within the frame and no edges or corners are cut off. The item is presented in a clean, perfectly symmetrical and de-wrinkled true flat lay, where the garment is spread completely flat on the surface. The background is a seamless, solid light gray studio backdrop with a specific hex code of #D7D7D7. A subtle, soft drop shadow is cast directly behind the item with no horizontal offset (x=0) and no vertical offset (y=0), giving the item a clean, distinct lift from the surface.

Crucially, all authentic brand details and logos must be preserved as pristine, identical elements; there must be zero hallucination, addition, or alteration of any logos, text, or symbols.

Studio lighting is bright, even, and diffused, provided by large softboxes to eliminate harsh reflections and provide clean details of the fabric structure and all intrinsic details. The focus is sharp across the entire garment, capturing intricate texture without being washed out. The presentation is immaculately clean and premium.
"""

# Your GitHub repo
GITHUB_REPO = 'michellefleek/garment-images'   # owner/repo
GITHUB_BRANCH = 'main'
INPUT_PATH = 'input'                            # folder with your source images
OUTPUT_PATH = 'output'                          # folder where results are saved

# The nano banana model
MODEL = 'gemini-2.5-flash-image'

# 🧪 TEST MODE: only process this many images (great for a first run).
# Set to None to process ALL images in the input folder.
TEST_LIMIT = 5

print('✅ Settings loaded.')

## Step 5 — Run the batch
Reads each image from `input/`, beautifies it, and saves it into `output/` in your repo.

Safe to stop and re-run — images already in `output/` are skipped.

In [ ]:
import time, io, base64, json, requests
from PIL import Image
from google import genai

client = genai.Client(api_key=API_KEY)
GH = {'Authorization': f'Bearer {GITHUB_TOKEN}', 'Accept': 'application/vnd.github+json'}
IMAGE_EXTS = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')
MAX_RETRIES = 4

def list_files(path):
    url = f'https://api.github.com/repos/{GITHUB_REPO}/contents/{path}?ref={GITHUB_BRANCH}'
    r = requests.get(url, headers=GH)
    if r.status_code == 404:
        return []
    r.raise_for_status()
    return [it for it in r.json() if it['type'] == 'file']

def beautify(img_bytes):
    img = Image.open(io.BytesIO(img_bytes))
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.models.generate_content(model=MODEL, contents=[PROMPT, img])
            for part in resp.candidates[0].content.parts:
                if getattr(part, 'inline_data', None) and part.inline_data.data:
                    return part.inline_data.data  # beautified image bytes (PNG)
            return None  # no image returned (e.g. safety block)
        except Exception as e:
            wait = min(2 ** attempt, 30)
            print(f'   ⚠️  attempt {attempt} failed ({e}); retrying in {wait}s...')
            time.sleep(wait)
    return None

def save_to_github(name, data_bytes):
    api = f'https://api.github.com/repos/{GITHUB_REPO}/contents/{OUTPUT_PATH}/{name}'
    existing = requests.get(f'{api}?ref={GITHUB_BRANCH}', headers=GH)
    sha = existing.json().get('sha') if existing.status_code == 200 else None
    payload = {'message': f'Add beautified {name}', 'branch': GITHUB_BRANCH,
               'content': base64.b64encode(data_bytes).decode()}
    if sha:
        payload['sha'] = sha  # update in place if it already exists
    r = requests.put(api, headers=GH, data=json.dumps(payload))
    r.raise_for_status()

inputs = [it for it in list_files(INPUT_PATH) if it['name'].lower().endswith(IMAGE_EXTS)]
already_done = {it['name'] for it in list_files(OUTPUT_PATH)}
if TEST_LIMIT:
    inputs = inputs[:TEST_LIMIT]
    print(f'🧪 TEST MODE: only processing the first {TEST_LIMIT} image(s).')
print(f'Found {len(inputs)} image(s) to process.\n')

done = skipped = failed = 0
for i, it in enumerate(inputs, 1):
    name = it['name']
    out_name = name.rsplit('.', 1)[0] + '.png'
    if out_name in already_done:
        skipped += 1
        print(f'[{i}/{len(inputs)}] ⏭️  {name} (already in output/)')
        continue
    print(f'[{i}/{len(inputs)}] 🎨 {name} ...')
    img_bytes = requests.get(it['download_url']).content
    out = beautify(img_bytes)
    if out:
        save_to_github(out_name, out)
        done += 1
        print(f'          ✅ saved to output/{out_name}')
    else:
        failed += 1
        print('          ❌ could not beautify this one (skipping)')
    time.sleep(1)  # gentle pacing

print(f'\n🏁 Finished. {done} new, {skipped} already done, {failed} failed.')
print(f'Results are in: https://github.com/{GITHUB_REPO}/tree/{GITHUB_BRANCH}/{OUTPUT_PATH}')